# Week 20: Advanced Computer Vision, 3D Vision & NeRFs
## Notebook 04: 3D Gaussian Splatting Foundations from Scratch

### 1. Conceptual Intuition & Real-World Use Case
While NeRF achieved stunning visual quality, its ray-marching volume rendering is too slow for real-time interactive rendering (>60 FPS).
In SIGGRAPH 2023, Kerbl et al. introduced **3D Gaussian Splatting**:
- Models 3D scenes as millions of parameterized **3D Gaussians (ellipsoidal particles)**.
- Each Gaussian has: 3D position $\mu$, 3D covariance $\Sigma$ (scale $S$ + rotation quaternion $R$), opacity $\alpha$, and spherical harmonic color coefficients.
- **Projects 3D Gaussians into 2D camera screen space** and rasterizes via fast GPU tile sorting at **100+ FPS**!

---

### 2. Mathematical Foundations: 3D to 2D Covariance Projection
A 3D Gaussian is defined as:
$$G(x) = \exp\left( -\frac{1}{2} (x - \mu)^T \Sigma^{-1} (x - \mu) \right)$$

To ensure $\Sigma$ is physically valid (Positive Semi-Definite):
$$\Sigma = R S S^T R^T$$
where $S$ is a 3D scaling vector and $R$ is a rotation matrix derived from a unit quaternion.

#### Projecting 3D Covariance to 2D Screen Space (Zwicker et al. EWA Splatting):
Given camera viewing transformation $W$ and projective Jacobian $J$:
$$\Sigma_{\text{2D}} = J W \Sigma W^T J^T$$
The resulting $2 \times 2$ matrix defines the 2D ellipse rendered onto the camera screen.

In [ ]:
import numpy as np

# Step 1: Define a 3D Gaussian Particle
mu_3d = np.array([0.0, 0.0, 3.0])  # Centered at (0, 0, 3 meters forward)
scale = np.array([0.2, 0.1, 0.4])   # Semi-axes lengths (S)

# Rotation Matrix R (Rotation around Y-axis by 45 degrees)
theta = np.pi / 4.0
R = np.array([
    [np.cos(theta),  0, np.sin(theta)],
    [0,              1, 0],
    [-np.sin(theta), 0, np.cos(theta)]
])

# Compute 3D Covariance Matrix Sigma = R * S * S^T * R^T
S = np.diag(scale)
Sigma_3d = R @ S @ S.T @ R.T

print("--- 3D Gaussian Representation ---")
print("Position mu:\n", mu_3d)
print("3D Covariance Matrix Sigma (3x3):\n", np.round(Sigma_3d, 4))

# Step 2: Project 3D Covariance onto 2D Camera Screen
# Pin-hole camera projection Jacobian J at depth z=3.0 (focal length f = 500)
f = 500.0
z = mu_3d[2]
J = np.array([
    [f / z, 0,     -(f * mu_3d[0]) / (z ** 2)],
    [0,     f / z, -(f * mu_3d[1]) / (z ** 2)]
])

# Camera view matrix W (Identity for world-aligned camera)
W = np.eye(3)

# Project to 2D Screen Space: Sigma_2D = J * W * Sigma * W.T * J.T
Sigma_2d = J @ W @ Sigma_3d @ W.T @ J.T

# Add low-pass filter (0.3 pixel blur) to prevent aliasing
Sigma_2d += np.eye(2) * 0.3

print("\n--- Projected 2D Screen Space Gaussian ---")
print("Projected Screen Covariance Matrix (2x2 pixels):\n", np.round(Sigma_2d, 2))

# Compute 2D Ellipse Eigenvalues (Pixel extents on screen)
eigenvals_2d = np.linalg.eigvals(Sigma_2d)
pixel_radius = 3.0 * np.sqrt(eigenvals_2d) # 3-sigma bounding box
print(f"Projected 2D Pixel Radii (3-sigma extent): {pixel_radius[0]:.1f}px by {pixel_radius[1]:.1f}px")

### 3. Step-by-Step Code Walkthrough
1. **Physically Valid Covariance:** Decomposing $\Sigma = R S S^T R^T$ guarantees positive semi-definiteness during gradient updates.
2. **EWA Splatting Projection:** The Jacobian $J$ maps 3D metric ellipsoids into 2D pixel coordinates on the camera sensor plane.
3. **Real-Time Splatting:** A $14 \times 50$ pixel 2D Gaussian splat is rasterized in microseconds using GPU parallel threads.

---

### 4. Industry Applications & Trade-offs
- **Real-Time 3D Scanning (Luma AI, Polycam):** Instant rendering of captured 3D scenes on mobile web browsers.
- **Trade-off:** High VRAM storage footprint: storing 5 million Gaussians requires ~1-2 GB of GPU RAM.

---

### 5. Top Interview Questions & Answers
**Q1: How does 3D Gaussian Splatting avoid the slow ray-marching bottlenecks of NeRF?**
> **Answer:** NeRF queries a neural network hundreds of times per pixel along every ray. 3D Gaussian Splatting pre-sorts 2D splats by depth into screen tiles and accumulates color using standard alpha blending, turning rendering into a memory-bound rasterization task evaluated in real time.

**Q2: How are Gaussians cloned or pruned during training?**
> **Answer:** Gaussians with position gradients above a threshold are densified: if they are small, they are cloned; if they are large, they are split into two smaller Gaussians. Gaussians with opacity $\alpha$ below a threshold are pruned.